# TB Portals - Kantipudi **A1** baseline (lung-seg + YOLOv5 lesion det + cavity)

Detection-based ALP = |lesion boxes ∩ MedSAM lung| / |lung|, + cavity classifier -> Timika. **This is the paper's WORST approach and the heaviest to run.** Also attach **tbx-11** (TBX11K) for the lesion detector. **Attach these Kaggle datasets before running:** `tb-portals-cxr-pngs`, `medsam-vit-b`.

## 0 - Clone the codebase

In [10]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

Already up to date.
repo ready at /kaggle/working/dl-project-codebase


## Install deps

In [11]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg", "ultralytics"], check=False)
print("deps installed")

deps installed


## Paths

Edit dataset slugs if yours differ.

In [12]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_a1"
os.makedirs(OUT_DIR, exist_ok=True)
TBX_ROOT  = "/kaggle/input/datasets/usmanshams/tbx-11/TBX11K"
YOLO_DIR  = f"{WORK}/tbx11k_yolo"
YOLO_BEST = f"{WORK}/yolo_runs/tbx11k/weights/best.pt"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
MEDSAM_CKPT:   /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth -> True
LUNG_DECODER:  /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt -> True


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [13]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [14]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main
argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))

[crops] device=cuda:Tesla T4
[crops] 0/5010 cached; generating the remaining 5010.
[crops] loaded fine-tuned lung decoder: /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt
[crops] 200/5010 (lung=200, fallback=0)
[crops] 400/5010 (lung=400, fallback=0)
[crops] 600/5010 (lung=600, fallback=0)
[crops] 800/5010 (lung=800, fallback=0)
[crops] 1000/5010 (lung=1000, fallback=0)
[crops] 1200/5010 (lung=1200, fallback=0)
[crops] 1400/5010 (lung=1400, fallback=0)
[crops] 1600/5010 (lung=1600, fallback=0)
[crops] 1800/5010 (lung=1800, fallback=0)
[crops] 2000/5010 (lung=2000, fallback=0)
[crops] 2200/5010 (lung=2200, fallback=0)
[crops] 2400/5010 (lung=2400, fallback=0)
[crops] 2600/5010 (lung=2600, fallback=0)
[crops] 2800/5010 (lung=2800, fallback=0)
[crops] 3000/5010 (lung=3000, fallback=0)
[crops] 3200/5010 (lung=3200, fallback=0)
[crops] 3400/5010 (lung=3400, fallback=0)
[crops] 3600/5010 (lung=3600, fallback=0)
[crops] 3800/5010 (lung=3800, fallback=0)
[

## 3 - Inspect TBX11K layout

The converter auto-discovers VOC XML. If this print shows a different structure (e.g. COCO JSON), tell me and I'll adjust the converter.

In [15]:
import os
print("TBX_ROOT exists:", os.path.isdir(TBX_ROOT))
if os.path.isdir(TBX_ROOT):
    for name in sorted(os.listdir(TBX_ROOT))[:20]:
        sub = os.path.join(TBX_ROOT, name)
        kids = sorted(os.listdir(sub))[:6] if os.path.isdir(sub) else "(file)"
        print(" ", name, "->", kids)

TBX_ROOT exists: True
  README.md -> (file)
  TBX11K_CVPR2020.pdf -> (file)
  annotations -> ['json', 'xml']
  code -> ['make_json_anno.py', 'make_json_anno.sh', 'pycococreatortools.py']
  imgs -> ['extra', 'health', 'sick', 'tb', 'test']
  lists -> ['TBX11K_train.txt', 'TBX11K_trainval.txt', 'TBX11K_val.txt', 'all_test.txt', 'all_train.txt', 'all_trainval.txt']
  teaser.jpg -> (file)


## 4 - Convert TBX11K -> YOLO format (paper's official split)

Uses `lists/TBX11K_train.txt` / `TBX11K_val.txt` intersected with the annotated TB images (reproduces the paper's detection split, ~511 train / ~128 val). Watch the printed counts; if `train`/`val` come out near 0, the list format differs - paste the output to me.

In [16]:
import sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from prepare_tbx11k_yolo import main as prep_main
prep_main(['--tbx-root', TBX_ROOT, '--out', YOLO_DIR,
           '--train-list', f"{TBX_ROOT}/lists/TBX11K_train.txt",
           '--val-list',   f"{TBX_ROOT}/lists/TBX11K_val.txt"])

[tbx11k] scanning /kaggle/input/datasets/usmanshams/tbx-11/TBX11K ...
[tbx11k] top-level entries: ['README.md', 'TBX11K_CVPR2020.pdf', 'annotations', 'code', 'imgs', 'lists', 'teaser.jpg']
[tbx11k] found 800 XML annotation files
[tbx11k] indexed 12279 image files
[tbx11k] TBX11K_train.txt: 6600 entries; sample=['tb/tb0005.png', 'tb/tb0007.png']
[tbx11k] TBX11K_val.txt: 1800 entries; sample=['tb/tb0003.png', 'tb/tb0004.png']
[tbx11k] official split: train=599 val=200 (skipped 0 annotated images not in train/val lists, e.g. test)
[tbx11k] train=599 val=200 images, 1211 lesion boxes, 0 unmatched.
[tbx11k] dataset config -> /kaggle/working/tbx11k_yolo/tbx11k.yaml


## 5 - Train YOLOv5 lesion detector on TBX11K (~30-60 min)

Paper used YOLOv5n. Epochs reduced from 1000 to 100 for Kaggle; increase if time allows.

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolov5nu.pt")
yolo.train(data=f"{YOLO_DIR}/tbx11k.yaml", epochs=100, imgsz=640, batch=16,
           project=f"{WORK}/yolo_runs", name="tbx11k", exist_ok=True)
print("best weights ->", YOLO_BEST)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.54 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/tbx11k_yolo/tbx11k.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hs

## 6 - A1 eval: detection ALP + cavity -> Timika (~1.5-2 h)

In [ ]:
from src.training.train_a1_detect import main as a1_main
a1_main(['--manifest', PAPER_MANIFEST, '--crops-dir', CROPS_DIR,
         '--yolo-weights', YOLO_BEST, '--medsam-ckpt', MEDSAM_CKPT, '--lung-decoder-ckpt', LUNG_DECODER,
         '--out-dir', OUT_DIR, '--held-outs','Romania','Moldova','Kazakhstan','--seeds','0','1','2',
         '--epochs','30','--batch-size','60','--accum-steps','5','--num-workers','2',
         '--cavity-no-lung-crop'])

[RESULT] A1 Romania seed=0  ALP_MAE=19.27  cavity_AUC=0.696  Timika_MAE=28.31 (20.22%) | their 23.83   Pearson=0.55 | their 0.59

===== A1  Romania  seed=1 =====
[A1][CAV] train=2915 val=733 test=220
  [CAV] epoch 00 train_ce=0.73303 val_ce=0.63022
  [CAV] epoch 01 train_ce=0.65858 val_ce=0.65370
  [CAV] epoch 02 train_ce=0.61889 val_ce=0.57618
  [CAV] epoch 03 train_ce=0.57996 val_ce=0.60969
  [CAV] epoch 04 train_ce=0.57040 val_ce=0.60092
  [CAV] epoch 05 train_ce=0.53805 val_ce=0.56527
  [CAV] epoch 06 train_ce=0.53481 val_ce=0.55908
  [CAV] epoch 07 train_ce=0.51512 val_ce=0.68469
  [CAV] epoch 08 train_ce=0.51777 val_ce=0.57157
  [CAV] epoch 09 train_ce=0.50055 val_ce=0.57543
  [CAV] epoch 10 train_ce=0.46080 val_ce=0.55260
  [CAV] epoch 11 train_ce=0.46393 val_ce=0.60109
  [CAV] epoch 12 train_ce=0.45122 val_ce=0.89820
  [CAV] epoch 13 train_ce=0.46132 val_ce=0.61936
  [CAV] epoch 14 train_ce=0.40543 val_ce=0.58140
  [CAV] epoch 15 train_ce=0.42930 val_ce=0.58963
  [CAV] epoch 16

## 7 - Save outputs

In [ ]:
!cd /kaggle/working && zip -j results_a1.zip checkpoints/paper_a1/results_a1.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q checkpoints_a1.zip checkpoints/paper_a1
print("Saved: results_a1.zip, checkpoints_a1.zip")